# 🔬 Ollama 架构深度剖析

> **核心命题**：Ollama 在 llama.cpp 之上加了什么，让本地 LLM 部署从「开发者工具」变成「人人都能用」？

Ollama 不是推理引擎，它是 llama.cpp 的**运维层 + 分发层**。理解它的价值在于理解「如何把一个 C++ 推理库
变成一个产品」。

## 架构全景

```
┌──────────────────────────────────────────────────────────────┐
│                      Ollama 架构                              │
├──────────────────────────────────────────────────────────────┤
│                                                               │
│  ┌─────────────────────────────────────────────────────────┐ │
│  │                   CLI / API Layer                        │ │
│  │  ollama run llama3    ollama pull mistral                │ │
│  │  ollama serve          /api/chat  (OpenAI compat)        │ │
│  └─────────────────────────┬───────────────────────────────┘ │
│                            │                                  │
│  ┌─────────────────────────▼───────────────────────────────┐ │
│  │                   Server (Go)                            │ │
│  │  • HTTP server (net/http)                                │ │
│  │  • 模型注册表 + Modelfile 解析                           │ │
│  │  • 模型拉取 (Registry → 本地 blob store)                 │ │
│  │  • 并发请求调度 (semaphore-based queue)                  │ │
│  │  • GPU 层检测 (CUDA/Metal/CPU)                           │ │
│  └─────────────────────────┬───────────────────────────────┘ │
│                            │  CGo / subprocess               │
│  ┌─────────────────────────▼───────────────────────────────┐ │
│  │                 llama.cpp (C++)                          │ │
│  │  • 实际推理执行                                          │ │
│  │  • llama.cpp 以动态库 (.so/.dylib) 形式链接              │ │
│  │  • 或通过子进程通信（多 GPU 场景）                       │ │
│  └─────────────────────────────────────────────────────────┘ │
│                                                               │
└──────────────────────────────────────────────────────────────┘
```

## 1. 核心创新：Modelfile

Ollama 最大的创新不是技术上的，而是用户体验上的——**Modelfile**。

```dockerfile
# Modelfile 示例 — 类比 Dockerfile
FROM llama3.2                    # 基础模型 (GGUF)
PARAMETER temperature 0.7        # 推理参数
PARAMETER top_p 0.9
PARAMETER stop "<|im_end|>"      # 停止词

SYSTEM """You are a helpful coding assistant.
Always respond in Chinese."""   # System prompt 写入 GGUF metadata

# 自定义 chat template (Jinja2)
TEMPLATE """{{ if .System }}<|system|>{{ .System }}<|end|>
{{ end }}{{ if .Prompt }}<|user|>{{ .Prompt }}<|end|>
{{ end }}<|assistant|>"""
```

Modelfile 的实际作用：
1. `ollama create mymodel -f Modelfile` → 生成一个新的 GGUF blob（写入了新的 metadata）
2. 底层的 GGUF metadata 被修改，加入了 system prompt、template、parameters
3. 推理时 llama.cpp 直接从 GGUF metadata 读取这些配置

```
ollama create 的内部流程:
  Modelfile 解析 → 读取基础模型的 GGUF
  → 修改 metadata (添加 system, template, stop tokens, params)
  → 生成新的 GGUF blob (sha256 hash)
  → 注册到本地模型列表 (manifest)
```

## 2. 模型分发：Registry + Blob Store

Ollama 的模型存储借鉴了 Docker 的分层设计。

```
~/.ollama/
├── models/
│   └── manifests/
│       └── registry.ollama.ai/
│           └── library/
│               └── llama3.2/
│                   └── 3b          ← manifest JSON {layers: [{digest: sha256:...}]}
├── blobs/
│   ├── sha256-abc123...            ← GGUF 模型文件 (~2GB)
│   ├── sha256-def456...            ← Modelfile 生成的配置层 (~1KB)
│   └── sha256-ghi789...            ← 另一个模型的共享层
```

**关键设计**：
- 每个模型层是 content-addressed (sha256)，天然支持去重和缓存
- 不同模型可以共享相同的 GGUF base layer（类似 Docker 的 layer 共享）
- `ollama pull` 支持断点续传和增量更新

## 3. 并发调度：Semaphore Queue

Ollama 在并发请求时的调度策略——简单但有效。

```go
// server/routes.go (简化)
type Server struct {
    // 信号量限制并发请求数
    parallelSlots chan struct{}  // buffered channel, 默认大小为 numGPUs

    // 模型加载管理
    loadedModels map[string]*LoadedModel
}

func (s *Server) serveChat(w http.ResponseWriter, r *http.Request) {
    // 1. 获取信号量 — 如果满了就排队等待
    slot := <-s.parallelSlots
    defer func() { s.parallelSlots <- slot }()  // 释放

    // 2. 加载模型（如果还没加载）
    // llama.cpp 的模型是进程级单例
    model := s.getOrLoadModel(modelName)

    // 3. 执行推理
    result := model.Predict(prompt)

    // 4. 流式返回
    for token := range result {
        w.Write(token)
    }
}
```

**关键限制**：
- llama.cpp 是进程内单例，一个进程只能有一个活跃的模型
- 多模型并发需要启动多个 llama.cpp 进程
- Ollama 通过 `parallelSlots` 的计数器来控制并发
- 请求在 Go 层排队，llama.cpp 层串行执行

对比 vLLM 的 Continuous Batching：
```
Ollama 并发模型:
  Request 1: [████████████████████████████████]  ← 占满整个推理时间
  Request 2:                                 [██████████]  ← 阻塞等待
  Request 3:                                            [██]  ← 阻塞等待

vLLM 并发模型:
  Step 0: [R1█] [R2█] [R3█]          ← 三个请求交替执行
  Step 1: [R1█] [R2█] [R3█]          ← 同一 batch
  Step 2: [R1█]        [R3█]         ← R2 完成，立即释放
```

Ollama 的选择是务实的：**并发少时不值得做 Continuous Batching**，额外的调度开销反而降低延迟。

## 4. GPU 层检测与调度

Ollama 的 GPU 检测简单直接：

```go
// gpu/gpu.go (简化)
func GetGPUInfo() GPUInfo {
    // 1. 尝试 CUDA
    if cudaLib := findCUDALibrary(); cudaLib != "" {
        return GPUInfo{Library: "cuda", VRAM: cudaMemInfo()}
    }
    // 2. 尝试 Metal (macOS)
    if runtime.GOOS == "darwin" {
        return GPUInfo{Library: "metal", VRAM: metalRecommendedVRAM()}
    }
    // 3. 回退到 CPU
    return GPUInfo{Library: "cpu"}
}
```

**数量化决策**：Ollama 将 GPU 层数（`num_gpu`）设置为自动推断：
- NVIDIA GPU: `num_gpu = min(totalLayers, floor(VRAM_free / layer_size))`
- Apple Silicon: 所有层都放 GPU（统一内存架构不需要拷贝）
- CPU only: `num_gpu = 0`

## 5. 关键设计权衡

### Go + CGo：为什么选 Go？

Ollama 的大部分代码（~80%）是 Go 写的：
- **goroutine** 天然适合 HTTP server + 并发调度
- **编译成单一二进制** — 用户 `curl | sh` 一键安装
- **CGo** 链接 llama.cpp — 编译时复杂但运行时高效

代价：CGo 交叉编译是噩梦，Ollama 的 CI 矩阵巨大（OS×Arch×GPU 后端）。

### 为什么不做 Continuous Batching？

Ollama 的官方立场：**不适合本地场景**。

- CB 的调度器和块管理增加延迟 ~5-10ms
- 本地通常 1-2 个并发，CB 的收益为零
- 如果用户需要高并发，应该直接用 vLLM/TGI

Ollama 3.0 路线图中有「实验性 multi-user scheduling」，但不作为核心特性。

## 6. Ollama vs llama.cpp：谁做了什么

| 功能 | llama.cpp 提供 | Ollama 添加 |
|------|--------------|------------|
| 推理引擎 | ggml + llama.cpp | 无 |
| 模型下载 | 无 | Registry + Pull + 断点续传 |
| 模型管理 | 无 | list / rm / cp / push |
| Modelfile | 无 | ✅ 核心创新 |
| HTTP API | llama-server (基础) | Ollama API (完整) |
| OpenAI 兼容 | 实验性 | ✅ 开箱即用 |
| 多平台安装 | 需编译 | `curl | sh` 一键 |
| 量化 | K-quant | 透传，无增强 |

## 下一步

- → 返回 `../00-overview.ipynb` 查看服务端框架对比
- → `../server/04-tensorrt-llm-deep-dive.ipynb`：了解编译优化的极限
- → `../server/03-vllm-deep-dive.ipynb`：理解 PagedAttention 为什么比 Ollama 的调度高效